This is improved from hexner_test_differentiable_openloop.ipynb in the following ways:

1. P2 uses a pure best response
2. Compute the exact minimax loss for small I and K
3. P1 uses a nested optimizer: for every update on policy distribution, optimize policy prototypes (actions) 

In [ ]:
# ==============================================================
#  Cell 0 : setup.py
#  --------------------------------------------------------------
#  Global imports, device configuration, physical / game constants,
#  and DS-GDA hyper-parameters.
# ==============================================================

import math, random, os, sys, copy, pathlib, itertools
from typing import Dict, Tuple, Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor

# -----------------------------------------------------------------
#  Device switch (CPU for quick debugging, GPU for full training)
# -----------------------------------------------------------------
USE_GPU   = torch.cuda.is_available()
device    = torch.device("cuda" if USE_GPU else "cpu")
print("Running on:", device)

# -----------------------------------------------------------------
#  Physical / game constants
# -----------------------------------------------------------------
τ         = 1./4                      # time-step (s)
T         = 1.0                       # horizon   (s)
K         = int(T / τ)                # number of steps
I         = 2                         # #targets  (fixed)

BOX_POS   = 2.0                       # |position| ≤ 2  m
BOX_VEL   = 2.0                       # |velocity| ≤ 1  m/s
BOX_ACC   = 4.0                       # |accel|    ≤ 2  m/s²   (action box)

Z_TARGETS = torch.tensor([[0.0,  1.0, 0.0, 0.0],
                          [0.0, -1.0, 0.0, 0.0]], device=device)  # (I,4)

Kmat      = torch.diag(torch.tensor([1., 1., 0., 0.], device=device))
R1        = torch.diag(torch.tensor([0.05, 0.025], device=device))
R2        = torch.diag(torch.tensor([0.05, 0.100], device=device))

# -----------------------------------------------------------------
#  Observation feature dimension (t + state + belief-coord)
# -----------------------------------------------------------------
BELIEF_DIM = 1                        # simplex (I-1) since I = 2
FEAT_DIM   = 1 + 8 + BELIEF_DIM       # = 10

# -----------------------------------------------------------------
#  DS-GDA hyper-parameters (default – will be overridden in main)
# -----------------------------------------------------------------
LR_P1      = 1e-2                     # initial LR for player-1 network
LR_P2      = 1e-3                     # initial LR for player-2 network
BATCH_SIZE = 16
EPOCHS     = 200000
MOMENTUM   = 0.6                      # β in DS-GDA paper       cut down from 0.9 to address limit-cycle
C_SQUARE   = 10.0                      # C² clipping parameter
SEED       = 4321
TEMPERATURE = 1.0                     # temperature for ZGR: higher = softer
NOISE_SCALE = -10.0                     # noise scale for action: more negative = less noise

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# -----------------------------------------------------------------
#  Utility: belief to feature (Δ¹ simplex → ℝ¹)
# -----------------------------------------------------------------
def belief_coord(p: Tensor) -> Tensor:
    """
    Convert p ∈ Δ² to its single free coordinate (p₁).
    p : (B,2) → (B,1)
    """
    return p[..., :1]

In [15]:
# ==============================================================
#  Cell 1 : diff_env.py
#  --------------------------------------------------------------
#  Differentiable rollout of Hexner's game.
#  Usage:
#     env = HexnerDiffEnv(batch_size=512)
#     loss, traj = env.rollout(p1_net, p2_net)
#     loss.backward()               # hits both networks
# ==============================================================

from typing import List


P0 = torch.full((1, I), 1.0 / I, device=device)

class HexnerDiffEnv:
    """
    Differentiable environment for Hexner's 2-player, 0-sum game.
    Everything is torch-tensor math on `device`; no @torch.no_grad.
    """
    def __init__(self, batch_size: int = 1):
        self.B = batch_size
        self.reset()

    # ----------------------------------------------------------
    def _running_loss(self, u1: Tensor, u2: Tensor) -> Tensor:
        """
        l(u1,u2) * τ   (B,)  positive for P1, negative for P2
        """
        u1_cost = (u1.unsqueeze(1) @ R1 @ u1.unsqueeze(-1)).squeeze(-1).squeeze(-1)
        u2_cost = (u2.unsqueeze(1) @ R2 @ u2.unsqueeze(-1)).squeeze(-1).squeeze(-1)
        return 0.5 * (u1_cost - u2_cost) * τ

    # ----------------------------------------------------------
    def _terminal_loss(self) -> Tensor:
        """
        g_i(x_K)   (B,)
        """
        x1 = self.x[:, 0:4]
        x2 = self.x[:, 4:8]
        del1 = x1 - Z_TARGETS[self.i_star]         # (B,4)
        del2 = x2 - Z_TARGETS[self.i_star]
        term = 0.5 * ((del1 @ Kmat * del1).sum(-1) -
                      (del2 @ Kmat * del2).sum(-1))
        return term

    # ----------------------------------------------------------
    def step_dynamics(self, u1: Tensor, u2: Tensor):
        """
        Second-order integrator:
            p ← p + v τ + ½ a τ²
            v ← v + a τ
        """
        pos1, vel1 = self.x[:, 0:2], self.x[:, 2:4]
        pos2, vel2 = self.x[:, 4:6], self.x[:, 6:8]

        pos1_new = (pos1 + vel1 * τ + 0.5 * u1 * τ**2).clamp(-BOX_POS, BOX_POS)
        vel1_new = (vel1 + u1 * τ).clamp(-BOX_VEL, BOX_VEL)
        pos2_new = (pos2 + vel2 * τ + 0.5 * u2 * τ**2).clamp(-BOX_POS, BOX_POS)
        vel2_new = (vel2 + u2 * τ).clamp(-BOX_VEL, BOX_VEL)

        self.x = torch.cat([pos1_new, vel1_new, pos2_new, vel2_new], dim=-1)
        self.t = self.t + τ

    # ----------------------------------------------------------
    def bayes_update(self, p, A, j):
        """
        Differentiable Bayes update of public belief  p_{t+1}.
        A : (B,I,I)   row-normalised matrix from P1 policy
        j : (B,)      component indices (no grad)
        """
        B = p.shape[0]
        idx = torch.arange(B, device=A.device)
        Aj  = A[idx, :, j]                       # (B, I)  column j
        numer = Aj * p                          # (B,I)
        denom = numer.sum(-1, keepdim=True).clamp_min(1e-8)
        return numer / denom                    # (B,I)

    # ----------------------------------------------------------
    def rollout(self, p1_net, p2_net):
        """
        Run K time-steps.
        Returns scalar loss  L(θ,φ) (mean over batch)  and a trajectory list.
        """
        running_costs: List[Tensor] = []

        for k in range(K):
            # -------- build observation dict (all require_grad=False) ----
            obs = {
                "t": self.t.detach(),      # detach – treat as constant
                "x": self.x,
                "p": self.p
            }

            # -------- forward through policies --------------------------
            u1, misc1 = p1_net.action_only(obs, self.i_star)   # (B,2)
            u2, misc2 = p2_net.action_only(obs)                # (B,2)

            # -------- continuous dynamics -------------------------------
            self.step_dynamics(u1, u2)

            # -------- running loss & record -----------------------------
            running_costs.append(self._running_loss(u1, u2))

            # -------- Bayes belief update -------------------------------
            self.p = self.bayes_update(self.p, misc1["A"], misc1["j"])

            # -------- store trajectory for visualisation ----------------
            self.traj.append({
                "p1_xy": self.x[:, 0:2].detach().cpu(),
                "p2_xy": self.x[:, 4:6].detach().cpu(),
                "belief": self.p.detach().cpu(),
            })

        total_running = torch.stack(running_costs, dim=0).sum(0)   # (B,)
        total_loss = (total_running + self._terminal_loss()).mean()  # scalar
        return total_loss, self.traj
    
    def reset(self):
        """
        Re-initialise state, belief and timer, then return the initial
        public observation dict.  Matches the signature used by
        animate_episode().
        """
        # 1. (re)sample targets and initial belief
        p0 = P0
        self.i_star = torch.multinomial(p0, 1).squeeze(-1)   # (B,)
        self.p      = p0.clone()

        # 2. zero state & time  →  set initial coordinates
        self.x = torch.zeros(self.B, 8, device=device)
        self.x[:, 0] = -0.5          # P1  x-coord
        self.x[:, 4] = +0.5          # P2  x-coord
        self.t = torch.zeros(self.B, device=device)

        # 3. clear stored trajectory
        self.traj = []

        # 4. return public observation
        return {
            "t": self.t.detach(),
            "x": self.x.detach(),
            "p": self.p.detach(),
        }

In [16]:
# =============================================================
#  Cell 2 : networks_exact.py
# =============================================================
"""
Closed-loop deterministic policy for P2 that outputs a single acceleration
vector u₂ ∈ ℝ².  This is used in the enumerative (exact-loss) setting where
P1 still plays a mixed open-loop schedule but P2 reacts with a best response.
"""
import torch, torch.nn as nn, torch.nn.functional as F
import torch.distributions as D

TEMPERATURE  = 0.3
LOG_STD_CONST = -10.0   # fixed exploration σ  (rarely used in exact mode)

# ---------------------------------------------------------------------------
#  P1PolicyExact – thin wrapper around OLParams for visualisation / debugging
# ---------------------------------------------------------------------------
class P1PolicyExact(nn.Module):
    """
    Adapter so that visualise.py can call
        u, misc = p1_net.action_only(obs, i_star)
    even though P1 is represented by open-loop tensors (θ, μ) in `params`.
    """
    def __init__(self, params):
        super().__init__()
        self.params = params              # instance of OLParams

    def action_only(self, obs: dict, i_star: torch.Tensor):
        """
        obs     : {'t','x','p'} (only 't' is used to pick the step)
        i_star  : (B,) tensor of ints in {0,1}
        """
        # ----- indices -----------------------------------------------------
        B   = obs["x"].shape[0]
        k   = int((obs["t"] / τ).item())          # current time-step  0 … K−1
        trg = i_star.view(-1)                     # (B,)

        # ----- fetch θ row → probabilities ---------------------------------
        logits_row = self.params.th1[k][trg]      # (B,I)
        A_soft     = torch.softmax(
                              self.params.th1[k] / 0.3, dim=-1)    # (I,I)

        # ----- choose component j  (argmax of soft row) --------------------
        j = torch.argmax(logits_row, dim=-1)      # (B,)

        # ----- deterministic action  u₁ = μ[k, j] -------------------------
        mu_table = self.params.mu1[k]             # (I,2)
        u = mu_table[j]                           # (B,2)

        # ----- misc dict ---------------------------------------------------
        misc = {
            "A":  A_soft,          # full matrix for Bayes update
            "row": torch.softmax(logits_row, dim=-1).detach(),
            "j":   j.detach(),
            "mu":  mu_table.detach()
        }
        return u, misc
    

class P2BestResponse(nn.Module):
    """π₂ : (t , x , p) ↦ u₂  (deterministic)"""
    def __init__(self, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(FEAT_DIM, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),   nn.ReLU(),
            nn.Linear(hidden, 2),        # accel x,y
            nn.Tanh()
        )

    # ----- helper ----------------------------------------------------------
    def _embed(self, obs: dict):
        return torch.cat([
            obs["t"].unsqueeze(-1),      # (B,1)
            obs["x"],                    # (B,8)
            belief_coord(obs["p"])       # (B,1)  2-simplex ↦ barycentric coord
        ], dim=-1)

    # ----------------------------------------------------------------------
    @torch.enable_grad()
    def forward(self, obs: dict):
        """Returns u₂ (B,2) that already saturates the accel box."""
        return self.net(self._embed(obs)) * BOX_ACC

    # convenience – match .action_only() API -------------------------------
    def action_only(self, obs):
        u = self.forward(obs)
        return u, {"mu": u.detach()}    # keep same return signature

In [29]:
# =============================================================
#  Cell 3 : inner_opt_exact.py
# =============================================================
"""
For I = 2 and K ≤ 10 we can enumerate every joint sequence of discrete
actions j₀…j_{K−1}.  The number of paths is Iᴷ = 1024 at K = 10, which is
tractable.

Functions
---------
exact_loss(params, p2_net)
    Returns the *exact* game value (scalar tensor) – differentiable w.r.t.
    π-logits θ and prototype table μ.

inner_opt_mu(params, p2_net, n_steps=10, lr=2e-1)
    Runs SGD on every μ[k] row *before* the outer GDA update.
"""
import itertools, torch, torch.nn as nn

# ------------------------------------------------------------------ helpers
def all_sequences(K, I=2, device=device):
    """Tensor shape (S, K) with every int sequence, S = Iᴷ."""
    seq = torch.tensor(list(itertools.product(range(I), repeat=K)),
                       device=device, dtype=torch.long)
    return seq                                            # (S, K)

SEQ_CACHE = {}   # keyed by K → tensor (S,K)

def rollout_sequence(params, p2_net, seq, i_star):
    """
    Simulate one deterministic path.
    seq   : (K,) tensor of ints j_k
    i_star: 0 or 1  (player-1 type)
    Returns running+terminal loss (scalar).
    """
    env = HexnerDiffEnv(batch_size=1)
    env.i_star.fill_(i_star)
    env.p.copy_(P0)                                 # prior

    for k, j in enumerate(seq.tolist()):
        # ----- P1 deterministic action ----------------------------
        mu = params.mu1[k, j]                       # (2,)
        u1 = mu.unsqueeze(0)                        # (1,2)

        # ----- P2 deterministic best response --------------------
        u2, _ = p2_net.action_only({
            "t": env.t,     # keep gradient through time if needed
            "x": env.x,     # allow gradient through state
            "p": env.p      # crucial: do NOT detach belief
        })

        # dynamics + running cost
        env.step_dynamics(u1, u2)

        # Bayes update with soft A column
        logits_mat = params.th1[k]                  # (I,I)
        A_soft     = torch.softmax(logits_mat / 0.3, dim=-1)
        col_vec    = A_soft[:, j].unsqueeze(0)      # (1,I)
        env.p = (col_vec * env.p) / (col_vec*env.p).sum(-1, keepdim=True)

    return env._running_loss(u1, u2) + env._terminal_loss()


# ------------------------------------------------------------------ exact loss
def exact_loss(params, p2_net):
    """Enumerate i ∈ {0,1} and j-sequence; return E[L] (scalar)."""
    K   = params.K
    seq = SEQ_CACHE.setdefault(K, all_sequences(K))  # (S,K)
    S   = seq.shape[0]

    losses = []
    probs  = []

    with torch.enable_grad():
        for i_star in (0, 1):
            # soft rows for the current type
            pi_rows = torch.softmax(params.th1[:, i_star], dim=-1)     # (K, I)

            # gather π[k, j_k]  for every sequence  (shape  K × S)
            idx_k   = torch.arange(K, device=device).unsqueeze(1)      # (K,1)
            path_prob = pi_rows[idx_k, seq.T]                          # (K,S)

            # ∏_k π[k,j_k]  → (S,)
            path_prob = path_prob.log().sum(0).exp()                   # (S,)

            for s in range(S):
                L = rollout_sequence(params, p2_net, seq[s], i_star)
                losses.append(L)
                probs.append(path_prob[s])

    losses = torch.stack(losses)               # (2S,)
    prior_i = P0[0, i_star]
    probs  = torch.stack(probs) * prior_i          # prior p₀ = 0.5 #TODO: fix this line when prior is not [0.5, 0.5]
    return (losses * probs).sum()              # scalar


def inner_opt_mu(params, p2_net, steps=10, lr=2e-1, track=False):
    """
    Gradient-descent on every μ[k] row using exact_loss.
    Uses torch.autograd.grad to pull grads from params.mu1 into the local mu.
    Returns history of loss if track=True.
    """
    # clone the entire prototype table as a leaf
    mu = nn.Parameter(params.mu1.data.clone())
    opt = torch.optim.SGD([mu], lr=lr)
    history = []

    for it in range(steps):
        # 1) copy the candidate mu into params.mu1 for loss eval
        params.mu1.data.copy_(mu.data)

        # 2) compute the exact expected loss
        loss = exact_loss(params, p2_net)
        if track:
            history.append(loss.item())

        # 3) compute gradients of loss w.r.t. params.mu1
        grads = torch.autograd.grad(loss, params.mu1, retain_graph=False)[0]
        # grads has same shape as params.mu1

        # 4) assign to local mu.grad so optimizer can see it
        mu.grad = grads.detach()

        # --------------- debug convergence of actions ---------------
        Ey_grad = debug_grad_analytic(params, p2_net, t=3, j=0)
        # ------------------------------------------------------------
        
        # 5) step and clamp
        opt.step()
        mu.data.clamp_(-BOX_ACC, BOX_ACC)

        # 6) debug print
        # print(f" inner it={it:02d}  loss={loss.item():.4f}  "
        #       f"∥dL/dμ∥={mu.grad.norm().item():.3f}")

    # after inner loop, write back to params.mu1
    params.mu1.data.copy_(mu.data)
    return history if track else None


# ---------------------------------------------------------------
#  DEBUG block  – call once per inner iteration
# ---------------------------------------------------------------
def debug_grad_analytic(params, p2_net, t=3, j=0):
    """
    Print analytic vs autodiff gradient for the y-component of μ[t,j].
    Includes running-cost derivative 0.025*τ*a.
    """
    c = 0.5 * τ**2

    # ---- probability that type-0 picks prototype j --------------
    row0 = params.th1[t, 0].softmax(-1)        # (2,)
    p    = row0[j].item()                      # scalar

    # ---- compute E_y[y] and E_y[v] -------------------------------
    seq   = SEQ_CACHE[params.K]
    S     = seq.shape[0]
    Ey, Ev, W = 0.0, 0.0, 0.0

    for i, rows in enumerate((params.th1[:,0], params.th1[:,1])):
        rows = rows.softmax(-1)                # (K,2)
        w_seq = (rows[torch.arange(params.K).unsqueeze(1), seq.T]
                 .log().sum(0).exp())          # (S,)

        for s in range(S):
            env = HexnerDiffEnv(batch_size=1)
            env.i_star.fill_(i)
            env.p.copy_(P0.clone())

            for k in range(params.K):
                jj  = seq[s,k].item()
                u1  = params.mu1[k,jj].unsqueeze(0)
                u2,_= p2_net.action_only({"t": env.t,
                                         "x": env.x,
                                         "p": env.p})
                env.step_dynamics(u1, u2)

                # Bayes update
                Asoft = params.th1[k].softmax(-1)
                col   = Asoft[:, jj:jj+1].T
                env.p = (col * env.p) / (col * env.p).sum(-1, keepdim=True)

                if k == t:
                    w = 0.5 * w_seq[s]        # prior ½
                    Ey += w * env.x[0, 1]     # y-pos
                    Ev += w * env.x[0, 3]     # y-vel (index 3)
                    W  += w

    Ey /= W
    Ev /= W

    # ---- analytic gradient --------------------------------------
    a   = params.mu1[t, j, 1].item()          # current prototype y
    g_an= c * (0.5*(Ey + τ*Ev) + 0.5*c*a + 0.5*(1-2*p)) \
          + 0.025 * τ * a

    # ---- autodiff gradient --------------------------------------
    loss = exact_loss(params, p2_net)
    g_auto = torch.autograd.grad(loss, params.mu1,
                                 retain_graph=False)[0][t, j, 1].item()

    print(f"[t={t}, j={j}]  a={a:+.3f}  Ey={Ey:+.3f}  Ev={Ev:+.3f} "
          f"p={p:.3f}  g_an={g_an:+.4e}  g_auto={g_auto:+.4e}")

In [25]:
# =============================================================
#  Cell 4 : open_loop_exact.py   — exact-loss DSGDA with inner μ–optim
# =============================================================
import itertools, math, torch

class MomentumBuffer:
    def __init__(self, params, beta=0.9):
        self.params = params                     # ← add this line
        self.m      = [torch.zeros_like(p) for p in params]
        self.beta   = beta

    def update(self):
        for m, p in zip(self.m, self.params):
            if p.grad is not None:
                m.mul_(self.beta).add_(p.grad, alpha=1 - self.beta)

    def clip_(self, C):
        for m in self.m:
            n = m.norm()
            if n > C:
                m.mul_(C / n)

    def apply_step(self, params, lr, ascent=False):
        sign = +1 if ascent else -1
        for p, m in zip(params, self.m):
            p.data.add_(m, alpha=sign * lr)


class OLParams(nn.Module):
    """
    Holds player-1 outer variables:
        th1 : (K, I, I) logits   – softmax along last dim → π
        mu1 : (K, I, 2)          – prototype actions per row
    """
    def __init__(self, K, delta=0.2):
        super().__init__()
        # --- logits initialised with small bias -----------------
        bias = torch.tensor([[+delta, -delta],
                             [-delta, +delta]], dtype=torch.float32)
        self.th1 = nn.Parameter(bias.repeat(K, 1, 1) +
                                0.02 * torch.randn(K, I, I))
        # --- prototype table as before --------------------------
        self.mu1 = nn.Parameter(torch.randn(K, I, 2) * 1.0)
        self.K = K


# ---------------------------------------------------------------------------
class DSGDA_Exact:
    """
    • Player-1:  θ (π logits) are outer vars, μ rows re-optimised each step.
    • Player-2:  deterministic closed-loop best-response network.
    """
    def __init__(self, K=2,
                 lr_pi=1e-2, lr_p2=3e-3,
                 beta=0.6,
                 C2_p1=1.0, C2_p2=10.0):
        self.params  = OLParams(K).to(device)          # θ + μ
        self.p1 = P1PolicyExact(self.params)
        self.p2_net  = P2BestResponse().to(device)
        self.p2 = self.p2_net

        # parameter groups (skip frozen tensors)
        self.pi_vars = [self.params.th1]
        self.p2_vars = [p for p in self.p2_net.parameters()
                        if p.requires_grad]

        # momentum buffers
        self.buf_p1  = MomentumBuffer(self.pi_vars, beta)
        self.buf_p2  = MomentumBuffer(self.p2_vars, beta)

        # h-params
        self.lr_pi   = lr_pi
        self.lr_p2   = lr_p2
        self.C1      = math.sqrt(C2_p1)
        self.C2      = math.sqrt(C2_p2)

    # -----------------------------------------------------------------------
    def step(self, inner_steps=10, inner_lr=0.2):
        
        # ----------- debug action convergence -----------
        # -- show row probabilities (softmax(A)) at each t -----------------
        with torch.no_grad():
            for k in range(4):
                p_row0 = self.params.th1[k,0].softmax(-1).cpu().numpy()
                p_row1 = self.params.th1[k,1].softmax(-1).cpu().numpy()
                print(f"π[{k},0] {p_row0}   π[{k},1] {p_row1}")

        # -- show belief trajectory for one enumerated sequence ------------
        env = HexnerDiffEnv(batch_size=1)
        env.i_star.fill_(0)
        print("belief 0:", env.p.cpu().tolist())
        for k in range(4):
            j = torch.argmax(self.params.th1[k,0]).item()
            # hard column update
            A_soft = self.params.th1[k].softmax(-1)
            col = A_soft[:, j:j+1].T       # (1,2)
            env.p = (col*env.p)/(col*env.p).sum(-1, keepdim=True)
            print(f"t={k}  col={j}  belief={env.p.cpu().tolist()}")

        # -- gradient wrt μ at t=3 before inner step ----------------------
        loss = exact_loss(self.params, self.p2)
        gr = torch.autograd.grad(loss, self.params.mu1, retain_graph=False)[0]
        print("‖∂L/∂μ[3,0]‖", gr[3,0].norm().item(),
            "‖∂L/∂μ[3,1]‖", gr[3,1].norm().item())
        # -----------------------------------------


        # ---------- inner μ optimisation -----------------------------------
        hist = inner_opt_mu(self.params, self.p2_net,
                            steps=inner_steps, lr=inner_lr, track=True)
        
        # print("inner μ‐loss history:", hist)
        
        inner_start, inner_end = hist[0], hist[-1]

        # ---------- exact outer loss --------------------------------------
        loss = exact_loss(self.params, self.p2_net)

        # ---------- back-prop on θ & P2  -----------------------------------
        for p in itertools.chain(self.pi_vars, self.p2_vars):
            if p.grad is not None:
                p.grad.zero_()
        loss.backward()
        # print("‖dL/dθ‖ =", self.params.th1.grad.norm().item())

        # ---------- momentum update + clip ---------------------------------
        self.buf_p1.update(); self.buf_p2.update()
        self.buf_p1.clip_(self.C1); self.buf_p2.clip_(self.C2)

        # ---------- parameter steps ----------------------------------------
        self.buf_p1.apply_step(self.pi_vars, self.lr_pi, ascent=False)
        self.buf_p2.apply_step(self.p2_vars, self.lr_p2, ascent=True)

        return {
            "L":      loss.item(),
            "μ_start": inner_start,
            "μ_end":   inner_end,
            "g_pi":   torch.stack([m.norm() for m in self.buf_p1.m]).mean().item(),
            "g_p2":   torch.stack([m.norm() for m in self.buf_p2.m]).mean().item()
        }

In [19]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
import torch

def animate_episode(p1_net, p2_net, fps=5):
    env = HexnerDiffEnv(batch_size=1)
    obs = env.reset()
    i_star = env.i_star  # keep as tensor for action_only

    p_traj, t_traj, p1_xy, p2_xy = [], [], [], []

    print("\n========== DEBUG ROLL-OUT ==========")
    for step in range(K):
        # --- get actions & misc ------------------------------
        with torch.no_grad():
            if hasattr(p1_net, "action_only"):
                u1, m1 = p1_net.action_only(obs, i_star)
            else:
                u1 = p1_net(obs, i_star)
                m1 = {}

            if hasattr(p2_net, "action_only"):
                u2, m2 = p2_net.action_only(obs)
            else:
                u2 = p2_net(obs)
                m2 = {}

        # --- print diagnostics --------------------------------
        print(f"\n[t = {step*τ: .2f} s]")
        # P1
        row1 = m1.get("row", None)
        if row1 is not None:
            print("P1 dist (true row) :", row1.cpu().numpy())
        j1 = m1.get("j", None)
        if j1 is not None:
            print("P1 chosen idx      :", int(j1.item()))
        print("P1 action u        :", u1.squeeze(0).cpu().numpy())
        mu1 = m1.get("mu", None)
        if mu1 is not None:
            print("P1 μ table         :\n", mu1.cpu().detach().numpy())

        # P2
        # no more tuple errors: u2 is guaranteed a Tensor
        print("P2 action u        :", u2.squeeze(0).cpu().detach().numpy())
        row2 = m2.get("row", None)
        if row2 is not None:
            print("P2 dist            :", row2.cpu().numpy())
        mu2 = m2.get("mu", None)
        if mu2 is not None:
            print("P2 μ table / vec   :\n", mu2.cpu().detach().numpy())

        # --- advance env ----------------------------------------
        env.step_dynamics(u1, u2)

        # --- update obs -----------------------------------------
        obs = {
            "t": env.t.detach(),
            "x": env.x.detach(),
            "p": env.p.detach()
        }

        # --- collect positions & belief for plotting -----------
        p1_xy.append(env.x[0, 0:2].cpu().detach().numpy())
        p2_xy.append(env.x[0, 4:6].cpu().detach().numpy())
        p_traj.append(env.p[0, 0].item())
        t_traj.append((step + 1) * τ)

    print("====================================\n")

    # ---------- build the animation plots -----------------------
    # include initial point
    p_belief = np.array([0.5] + p_traj)
    times    = np.array([0.0] + t_traj)
    # initial positions
    p1_start = env.x[0, 0:2].cpu().detach().numpy()
    p2_start = env.x[0, 4:6].cpu().detach().numpy()
    p1_xy    = np.vstack([p1_start, *p1_xy])
    p2_xy    = np.vstack([p2_start, *p2_xy])

    fig, (ax0, ax1) = plt.subplots(2,1,figsize=(5,8),
                                   gridspec_kw={'height_ratios':[3,1]})
    ax0.set_xlim(-BOX_POS-.2, BOX_POS+.2)
    ax0.set_ylim(-BOX_POS-.2, BOX_POS+.2)
    ax0.set_aspect('equal')
    ax0.set_title("Hexner – trajectories")

    # plot targets
    for idx, tgt in enumerate(Z_TARGETS.cpu().numpy()):
        star_kw = dict(marker="*", ms=14,
                       color="black" if idx==i_star.item() else "grey")
        ax0.plot(tgt[0], tgt[1], **star_kw)

    p1_sc = ax0.scatter([],[], s=80, c="red")
    p2_sc = ax0.scatter([],[], s=80, c="blue")

    # belief subplot
    ax1.set_xlim(0, T)
    ax1.set_ylim(-.05, 1.05)
    ax1.set_xlabel("time (s)")
    ax1.set_ylabel("public belief p[0]")
    ax1.plot(times, p_belief, color="blue")
    ax1.axhline(y=float(i_star.item()==0),
                color="black", ls="--")
    ax1.set_title("Belief trajectory")

    def init():
        p1_sc.set_offsets(np.empty((0,2)))
        p2_sc.set_offsets(np.empty((0,2)))
        return p1_sc, p2_sc

    def update(frame):
        p1_sc.set_offsets(p1_xy[frame])
        p2_sc.set_offsets(p2_xy[frame])
        return p1_sc, p2_sc

    ani = animation.FuncAnimation(fig, update, frames=len(times),
                                  init_func=init, blit=True,
                                  interval=1000/fps)
    plt.close(fig)
    return HTML(ani.to_jshtml())

In [ ]:
from IPython.display import display

solver = DSGDA_Exact(K=K, lr_pi=LR_P1, lr_p2=LR_P2)
for epoch in range(EPOCHS):
    stats = solver.step()
    if epoch % 100 == 0:
        print(f"[{epoch:04d}]  L={stats['L']:.3f}  "
              f"μΔ={stats['μ_start']-stats['μ_end']:+.3e}  "
              f"gπ={stats['g_pi']:.3f}  gp2={stats['g_p2']:.2f}")
    if epoch % 100 == 0:
        print("🎞 visual check"); display(animate_episode(solver.p1, solver.p2))

π[0,0] [0.6025232  0.39747682]   π[0,1] [0.40629232 0.5937076 ]
π[1,0] [0.5941674 0.4058325]   π[1,1] [0.3953898  0.60461026]
π[2,0] [0.5988756 0.4011244]   π[2,1] [0.40711704 0.592883  ]
π[3,0] [0.5971202  0.40287977]   π[3,1] [0.40546075 0.5945393 ]
belief 0: [[0.5, 0.5]]
t=0  col=0  belief=[[0.5972580909729004, 0.402741938829422]]
t=1  col=0  belief=[[0.6902616024017334, 0.3097384572029114]]
t=2  col=0  belief=[[0.7662570476531982, 0.23374298214912415]]
t=3  col=0  belief=[[0.8284084796905518, 0.17159147560596466]]
‖∂L/∂μ[3,0]‖ 0.3977307975292206 ‖∂L/∂μ[3,1]‖ 0.42754852771759033
[t=3, j=0]  a=-0.473  Ey=+0.168  Ev=+0.109 p=0.597  g_an=-3.1753e-03  g_auto=+3.2489e-02
[t=3, j=0]  a=-0.479  Ey=+0.121  Ev=+0.039 p=0.597  g_an=-4.2193e-03  g_auto=+8.6185e-03
[t=3, j=0]  a=-0.481  Ey=+0.087  Ev=-0.013 p=0.597  g_an=-4.9752e-03  g_auto=-8.8415e-03
[t=3, j=0]  a=-0.479  Ey=+0.061  Ev=-0.050 p=0.597  g_an=-5.5107e-03  g_auto=-2.1388e-02
[t=3, j=0]  a=-0.475  Ey=+0.042  Ev=-0.078 p=0.597  g_a

π[0,0] [0.6022184 0.3977816]   π[0,1] [0.40617296 0.593827  ]
π[1,0] [0.5941054  0.40589455]   π[1,1] [0.39526445 0.6047356 ]
π[2,0] [0.5988395  0.40116048]   π[2,1] [0.4070889  0.59291106]
π[3,0] [0.5971202  0.40287977]   π[3,1] [0.40546075 0.5945393 ]
belief 0: [[0.5, 0.5]]
t=0  col=0  belief=[[0.5972070097923279, 0.40279296040534973]]
t=1  col=0  belief=[[0.6902616620063782, 0.3097383677959442]]
t=2  col=0  belief=[[0.766258716583252, 0.23374129831790924]]
t=3  col=0  belief=[[0.8284099102020264, 0.17159013450145721]]
‖∂L/∂μ[3,0]‖ 0.0646480843424797 ‖∂L/∂μ[3,1]‖ 0.057447649538517
[t=3, j=0]  a=-0.426  Ey=-0.002  Ev=-0.137 p=0.597  g_an=-6.4739e-03  g_auto=-4.7066e-02
[t=3, j=0]  a=-0.417  Ey=-0.004  Ev=-0.138 p=0.597  g_an=-6.4473e-03  g_auto=-4.7032e-02
[t=3, j=0]  a=-0.407  Ey=-0.006  Ev=-0.139 p=0.597  g_an=-6.4097e-03  g_auto=-4.6744e-02
[t=3, j=0]  a=-0.398  Ey=-0.007  Ev=-0.139 p=0.597  g_an=-6.3651e-03  g_auto=-4.6286e-02
[t=3, j=0]  a=-0.389  Ey=-0.008  Ev=-0.140 p=0.597  g_

π[0,0] [0.6035517  0.39644834]   π[0,1] [0.39914507 0.6008549 ]
π[1,0] [0.59262776 0.40737224]   π[1,1] [0.39161158 0.6083884 ]
π[2,0] [0.5962588 0.4037412]   π[2,1] [0.40329525 0.5967048 ]
π[3,0] [0.5971202  0.40287977]   π[3,1] [0.40546075 0.5945393 ]
belief 0: [[0.5, 0.5]]
t=0  col=0  belief=[[0.6019284129142761, 0.3980715572834015]]
t=1  col=0  belief=[[0.6958901286125183, 0.3041098713874817]]
t=2  col=0  belief=[[0.7718538045883179, 0.22814618051052094]]
t=3  col=0  belief=[[0.8328418135643005, 0.16715820133686066]]
‖∂L/∂μ[3,0]‖ 3.727970033651218e-05 ‖∂L/∂μ[3,1]‖ 3.727970033651218e-05
[t=3, j=0]  a=+0.009  Ey=-0.002  Ev=-0.148 p=0.597  g_an=-3.5798e-03  g_auto=+1.1858e-05
[t=3, j=0]  a=+0.009  Ey=-0.002  Ev=-0.148 p=0.597  g_an=-3.5800e-03  g_auto=+1.2863e-05
[t=3, j=0]  a=+0.009  Ey=-0.002  Ev=-0.148 p=0.597  g_an=-3.5801e-03  g_auto=+1.5486e-05
[t=3, j=0]  a=+0.009  Ey=-0.002  Ev=-0.148 p=0.597  g_an=-3.5802e-03  g_auto=+1.7315e-05
[t=3, j=0]  a=+0.009  Ey=-0.002  Ev=-0.148 p=0.

π[0,0] [0.60555947 0.3944406 ]   π[0,1] [0.39600223 0.60399777]
π[1,0] [0.5917652  0.40823478]   π[1,1] [0.38949296 0.6105071 ]
π[2,0] [0.59508646 0.40491346]   π[2,1] [0.40094495 0.5990551 ]
π[3,0] [0.5971202  0.40287977]   π[3,1] [0.40546075 0.5945393 ]
belief 0: [[0.5, 0.5]]
t=0  col=0  belief=[[0.6046152710914612, 0.3953847885131836]]
t=1  col=0  belief=[[0.6990960240364075, 0.30090391635894775]]
t=2  col=0  belief=[[0.7751947045326233, 0.22480526566505432]]
t=3  col=0  belief=[[0.8354800343513489, 0.1645200252532959]]
‖∂L/∂μ[3,0]‖ 1.815265932236798e-05 ‖∂L/∂μ[3,1]‖ 1.815265932236798e-05
[t=3, j=0]  a=+0.006  Ey=-0.001  Ev=-0.181 p=0.597  g_an=-3.7180e-03  g_auto=+1.6224e-05
[t=3, j=0]  a=+0.006  Ey=-0.001  Ev=-0.181 p=0.597  g_an=-3.7180e-03  g_auto=+1.6913e-05
[t=3, j=0]  a=+0.006  Ey=-0.001  Ev=-0.181 p=0.597  g_an=-3.7180e-03  g_auto=+1.7289e-05
[t=3, j=0]  a=+0.006  Ey=-0.001  Ev=-0.181 p=0.597  g_an=-3.7181e-03  g_auto=+1.7498e-05
[t=3, j=0]  a=+0.006  Ey=-0.001  Ev=-0.181 p=

π[0,0] [0.607107 0.392893]   π[0,1] [0.39399236 0.60600764]
π[1,0] [0.5914318  0.40856817]   π[1,1] [0.3884224  0.61157763]
π[2,0] [0.5945612 0.4054388]   π[2,1] [0.39988804 0.6001119 ]
π[3,0] [0.5971202  0.40287977]   π[3,1] [0.40546075 0.5945393 ]
belief 0: [[0.5, 0.5]]
t=0  col=0  belief=[[0.6064403057098389, 0.39355969429016113]]
t=1  col=0  belief=[[0.7011597156524658, 0.2988402545452118]]
t=2  col=0  belief=[[0.7772073745727539, 0.22279265522956848]]
t=3  col=0  belief=[[0.8370662927627563, 0.16293367743492126]]
‖∂L/∂μ[3,0]‖ 1.0199222742812708e-05 ‖∂L/∂μ[3,1]‖ 1.0199222742812708e-05
[t=3, j=0]  a=+0.003  Ey=-0.001  Ev=-0.182 p=0.597  g_an=-3.7342e-03  g_auto=+8.8848e-06
[t=3, j=0]  a=+0.003  Ey=-0.001  Ev=-0.182 p=0.597  g_an=-3.7341e-03  g_auto=+9.6671e-06
[t=3, j=0]  a=+0.003  Ey=-0.001  Ev=-0.182 p=0.597  g_an=-3.7341e-03  g_auto=+1.0174e-05
[t=3, j=0]  a=+0.003  Ey=-0.001  Ev=-0.182 p=0.597  g_an=-3.7341e-03  g_auto=+1.0174e-05
[t=3, j=0]  a=+0.003  Ey=-0.001  Ev=-0.182 p=0.5

π[0,0] [0.6083824 0.3916176]   π[0,1] [0.39277142 0.6072286 ]
π[1,0] [0.5914001 0.4085999]   π[1,1] [0.38794795 0.6120521 ]
π[2,0] [0.5943829 0.4056171]   π[2,1] [0.39951354 0.6004864 ]
π[3,0] [0.5971202  0.40287977]   π[3,1] [0.40546075 0.5945393 ]
belief 0: [[0.5, 0.5]]
t=0  col=0  belief=[[0.6076812744140625, 0.3923187553882599]]
t=1  col=0  belief=[[0.7024929523468018, 0.29750704765319824]]
t=2  col=0  belief=[[0.7784184217453003, 0.22158154845237732]]
t=3  col=0  belief=[[0.8380199074745178, 0.16198016703128815]]
‖∂L/∂μ[3,0]‖ 3.950096925109392e-06 ‖∂L/∂μ[3,1]‖ 3.950096925109392e-06
[t=3, j=0]  a=+0.001  Ey=-0.000  Ev=-0.178 p=0.597  g_an=-3.7260e-03  g_auto=+2.8498e-06
[t=3, j=0]  a=+0.001  Ey=-0.000  Ev=-0.178 p=0.597  g_an=-3.7260e-03  g_auto=+3.4049e-06
[t=3, j=0]  a=+0.001  Ey=-0.000  Ev=-0.178 p=0.597  g_an=-3.7260e-03  g_auto=+3.8184e-06
[t=3, j=0]  a=+0.001  Ey=-0.000  Ev=-0.178 p=0.597  g_an=-3.7260e-03  g_auto=+4.0568e-06
[t=3, j=0]  a=+0.001  Ey=-0.000  Ev=-0.178 p=0.597

π[0,0] [0.60922927 0.39077076]   π[0,1] [0.3920748  0.60792524]
π[1,0] [0.5914275  0.40857255]   π[1,1] [0.38772103 0.61227894]
π[2,0] [0.5943083  0.40569165]   π[2,1] [0.39939195 0.600608  ]
π[3,0] [0.5971202  0.40287977]   π[3,1] [0.40546075 0.5945393 ]
belief 0: [[0.5, 0.5]]
t=0  col=0  belief=[[0.608435869216919, 0.39156419038772583]]
t=1  col=0  belief=[[0.7032859921455383, 0.2967139482498169]]
t=2  col=0  belief=[[0.7791035771369934, 0.22089643776416779]]
t=3  col=0  belief=[[0.8385589122772217, 0.16144110262393951]]
‖∂L/∂μ[3,0]‖ 1.5938491060296656e-06 ‖∂L/∂μ[3,1]‖ 1.5938491060296656e-06
[t=3, j=0]  a=+0.001  Ey=-0.000  Ev=-0.176 p=0.597  g_an=-3.7187e-03  g_auto=+1.5162e-06
[t=3, j=0]  a=+0.001  Ey=-0.000  Ev=-0.176 p=0.597  g_an=-3.7188e-03  g_auto=+5.8860e-07
[t=3, j=0]  a=+0.001  Ey=-0.000  Ev=-0.176 p=0.597  g_an=-3.7188e-03  g_auto=-6.3330e-08
[t=3, j=0]  a=+0.001  Ey=-0.000  Ev=-0.176 p=0.597  g_an=-3.7188e-03  g_auto=-4.8801e-07
[t=3, j=0]  a=+0.001  Ey=-0.000  Ev=-0.176 

π[0,0] [0.6099217 0.3900783]   π[0,1] [0.3916304 0.6083696]
π[1,0] [0.5914369  0.40856302]   π[1,1] [0.38762075 0.61237925]
π[2,0] [0.5942617  0.40573823]   π[2,1] [0.39935312 0.6006469 ]
π[3,0] [0.5971202  0.40287977]   π[3,1] [0.40546075 0.5945393 ]
belief 0: [[0.5, 0.5]]
t=0  col=0  belief=[[0.6089764833450317, 0.3910234868526459]]
t=1  col=0  belief=[[0.7038167715072632, 0.2961832582950592]]
t=2  col=0  belief=[[0.779544472694397, 0.2204555869102478]]
t=3  col=0  belief=[[0.8389055728912354, 0.16109435260295868]]
‖∂L/∂μ[3,0]‖ 2.774243057501735e-06 ‖∂L/∂μ[3,1]‖ 2.774243057501735e-06
[t=3, j=0]  a=+0.001  Ey=-0.000  Ev=-0.174 p=0.597  g_an=-3.7108e-03  g_auto=+1.2629e-06
[t=3, j=0]  a=+0.001  Ey=-0.000  Ev=-0.174 p=0.597  g_an=-3.7107e-03  g_auto=+1.2852e-06
[t=3, j=0]  a=+0.001  Ey=-0.000  Ev=-0.174 p=0.597  g_an=-3.7107e-03  g_auto=+1.3039e-06
[t=3, j=0]  a=+0.001  Ey=-0.000  Ev=-0.174 p=0.597  g_an=-3.7107e-03  g_auto=+1.3076e-06
[t=3, j=0]  a=+0.001  Ey=-0.000  Ev=-0.174 p=0.597 

π[0,0] [0.6103993  0.38960075]   π[0,1] [0.39136425 0.6086358 ]
π[1,0] [0.5914719  0.40852806]   π[1,1] [0.38752034 0.6124796 ]
π[2,0] [0.59424335 0.40575662]   π[2,1] [0.39934462 0.6006554 ]
π[3,0] [0.5971202  0.40287977]   π[3,1] [0.40546075 0.5945393 ]
belief 0: [[0.5, 0.5]]
t=0  col=0  belief=[[0.6093246936798096, 0.39067524671554565]]
t=1  col=0  belief=[[0.7041878700256348, 0.29581218957901]]
t=2  col=0  belief=[[0.7798486351966858, 0.22015133500099182]]
t=3  col=0  belief=[[0.8391448259353638, 0.16085514426231384]]
‖∂L/∂μ[3,0]‖ 1.1799018011515727e-06 ‖∂L/∂μ[3,1]‖ 1.1799018011515727e-06
[t=3, j=0]  a=+0.000  Ey=-0.000  Ev=-0.172 p=0.597  g_an=-3.7046e-03  g_auto=+7.4506e-07
[t=3, j=0]  a=+0.000  Ey=-0.000  Ev=-0.172 p=0.597  g_an=-3.7046e-03  g_auto=+7.8976e-07
[t=3, j=0]  a=+0.000  Ey=-0.000  Ev=-0.172 p=0.597  g_an=-3.7046e-03  g_auto=+7.9349e-07
[t=3, j=0]  a=+0.000  Ey=-0.000  Ev=-0.172 p=0.597  g_an=-3.7046e-03  g_auto=+8.1584e-07
[t=3, j=0]  a=+0.000  Ey=-0.000  Ev=-0.172 p